# News alerts - Lane A (the real experience: you prompt, the agent builds)

**SISMID 2026 - Day 2, 11:00.** You drive a coding agent (Codex, Claude Code, or
Antigravity CLI). Paste each prompt, run the code, apply the check.


## About this data source

Four ways the world learns about an outbreak *before the data exists*, in order of how
easily we can use them:

- **GDELT** indexes world news in 100+ languages and gives it away through a free API with
  **no key**, which is why it is the one we scrape in class.
  **Explore:** <https://www.gdeltproject.org/>
- **HealthMap** reads news and official reports automatically and pins them on a world map.
  **Explore:** <https://www.healthmap.org/>
- **ProMED** is the opposite: real human experts posting curated outbreak reports since 1994.
  It flagged COVID about a day before the official notice. **Explore:**
  <https://promedmail.org/>
- **Media Cloud** is the scholars' media-analysis tool: **200M+ stories** in curated
  *collections* (national presses, country sets), built for studying *how* a topic is
  covered. Needs a **free API key**, so here it is a bonus rather than the main path.
  **Explore:** <https://search.mediacloud.org/>

> Open HealthMap and zoom to a region (every dot is a report), then read one ProMED post, to
> feel the difference between an automated map and a curated expert alert. Media Cloud shows
> you a third thing: attention over time, by outlet.

**The trade-off worth naming:** GDELT is *breadth with no key*; Media Cloud is *curation
with a key*. That is the same pattern as OpenSky vs OpenFlights, and Facebook posts vs the
Facebook symptom survey.


## Step 1: scan the news

> *Using the GDELT DOC 2.0 API (free, no key), find news articles from the past month*
> *that mention measles outbreaks. Return a tidy table with date, source country, domain,*
> *headline and URL, and report how many articles and the date range.*


In [ ]:
import pandas as pd, matplotlib.pyplot as plt, os, io, json
import urllib.request, urllib.parse

UA = {'User-Agent': 'SISMID2026-course/1.0 (your-email@example.com)'}

def cache_path(fname):
    for p in (f'../data/{fname}', f'data/{fname}', f'./{fname}'):
        if os.path.exists(p):
            return p
    return None

def fetch(url, timeout=120):
    return urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=timeout).read()

# ===== EDIT for your own disease / region =====
MY_QUERY = 'measles outbreak'
TIMESPAN = '24m'          # 1m = past month; also 1w, 3m ...
# =============================================

# --- Produced by the agent from the Plan A / Step 1 prompt ---
CACHE_FILE = 'gdelt_measles_articles.csv'
api = ('https://api.gdeltproject.org/api/v2/doc/doc'
       f'?query={urllib.parse.quote(MY_QUERY)}&mode=artlist&maxrecords=250'
       f'&format=json&timespan={TIMESPAN}')
try:
    arts = json.loads(fetch(api)).get('articles', [])
    news = pd.DataFrame([{'seendate': a.get('seendate',''),
                          'sourcecountry': a.get('sourcecountry',''),
                          'domain': a.get('domain',''),
                          'title': (a.get('title') or '').strip(),
                          'url': a.get('url','')} for a in arts])
    if news.empty: raise RuntimeError('no articles returned (rate-limited?)')
    news['date'] = pd.to_datetime(news['seendate'], format='%Y%m%dT%H%M%SZ', errors='coerce')
    news.to_csv(f'../data/{CACHE_FILE}', index=False)
except Exception as e:
    p = cache_path(CACHE_FILE)
    print('Live GDELT pull failed:', e, '-> cache', p)
    news = pd.read_csv(p, parse_dates=['date'])

print(f"{len(news)} articles, {news['date'].min()} to {news['date'].max()}")
news[['date','sourcecountry','title']].head(8)


## Step 2: emerging-outbreak watch

> *Tally the articles by source country and plot the top 12, then plot article volume*
> *per day. Summarise which countries are newly reporting activity.*

**Your check:** is a spike a real outbreak, or one wire story syndicated everywhere?


In [ ]:
# Agent's country tally + volume plot:
countries = news['sourcecountry'].replace('', 'unknown')

top12 = countries.value_counts().head(12)
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(top12.index[::-1], top12.values[::-1], color='#3B8A5B')
ax.set(xlabel='articles', title=f'News mentions of "{MY_QUERY}" by source country (top 12)')
fig.tight_layout(); plt.show()
display(top12)

daily = news.set_index('date').resample('D').size()
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(daily.index, daily.values, marker='o', ms=3, lw=1, color='#3B8A5B')
ax.set(ylabel='articles/day', title=f'Coverage volume: "{MY_QUERY}"')
fig.tight_layout(); plt.show()

# Newly reporting: countries whose coverage only appears in the most recent slice of
# the window, with no earlier mentions to compare against.
cutoff = news['date'].max() - pd.Timedelta(days=14)
recent_countries = set(countries[news['date'] > cutoff])
earlier_countries = set(countries[news['date'] <= cutoff])
newly_reporting = sorted(recent_countries - earlier_countries)
print(f"\nCountries newly reporting in the last 14 days (vs. {cutoff.date()} and earlier):")
print(', '.join(newly_reporting) if newly_reporting else '(none - all recent countries were already reporting)')



## Step 3: sanity-check and save

> *Report article count, unique domains, and how many duplicate headlines there are*
> *(syndication). Save a tidy CSV and print the five most recent headlines.*


In [ ]:
# Agent's checks + save:


## Bonus: Media Cloud (needs a free API key)

> *Media Cloud indexes 200M+ news stories in curated collections and has a Python client*
> *(`pip install mediacloud`). Using mediacloud.api.SearchApi with my MEDIACLOUD_API_KEY*
> *environment variable, pull stories from the last 30 days matching my query in the US*
> *national collection (id 34412234), and show publish date, outlet and headline. If the key*
> *is not set, skip cleanly and tell me GDELT already covered it.*

Get a free key by signing up at <https://search.mediacloud.org/>. Compare what Media Cloud
returns against GDELT: curated collections versus raw breadth.


In [ ]:
# Agent's Media Cloud pull (optional, needs MEDIACLOUD_API_KEY):


## Reflection

- News catches what has no data stream yet; it needs a human read.
- **Stretch:** swap in your own disease and region, and try `TIMESPAN='1w'`.
